# SegFormer + MaxSig — Stage-Placement Ablation

`SegFormer_maxsig.ipynb` applies a `MaxSigGate` (the same gated-residual `MaxSigLayerNormalized`
block used in the U-Net and TransUNet notebooks) right after selected stages of a from-scratch
Mix Vision Transformer (MiT) encoder. The encoder has 4 stages; for the `b0` variant their
channel widths are:

| stage index | channels | relative resolution |
|---|---|---|
| 0 | 32  | highest (H/4)  |
| 1 | 64  | H/8  |
| 2 | 160 | H/16 |
| 3 | 256 | lowest (H/32, bottleneck-like) |

The **default** configuration applies `MaxSigGate` at stages `(1, 2)`.

This notebook keeps that default run and adds the **two additional scenarios a reviewer is most
likely to ask for**, following the same logic already used for the U-Net and TransUNet ablations:

1. **Default (unchanged)** — MaxSig at stages `(1, 2)`
2. **MaxSig at Stage 0 only (32ch, highest resolution)** — reviewer framing: *"Did you test
   applying MaxSig at the earliest, highest-resolution encoder stage and find it hurt
   performance?"* — the SegFormer analogue of the Stage-0 (64ch) question raised for the plain
   U-Net.
3. **MaxSig at all four stages (0, 1, 2, 3)** — reviewer framing: *"Did you test adding MaxSig to
   every encoder stage, including the deepest, attention-heavy one, to see whether the benefit
   compounds or performance degrades?"*

Each scenario is its own cell and reports only: **best validation Dice**, **best epoch**, and
**test Dice** — the same reporting convention used in the other two ablation notebooks.


In [1]:
# ============================================================
# Config
# ============================================================
ROOT = '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/UnetMaxsig/compsitdataset/'

config = {
    "lr": 1e-4,
    "batch_size": 8,
    "epochs": 50,                 # lower this (e.g. 40-60) for a faster 3-run sweep if needed
    "segformer_variant": "b0",     # b0 (smallest/fastest) ... b5 (largest)
    "maxsig_kernel": 3,
    "gamma_init": 0.0,             # 0 => baseline SegFormer at init, MaxSig mixed in as it trains
}

RESULTS_DIR = ROOT + 'segformer_ablation_ckpts/'


In [2]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

from time import perf_counter
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, Dataset
from glob import glob
import albumentations as A
from albumentations.pytorch import ToTensorV2

from transformers import SegformerConfig, SegformerDecodeHead

torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


/home/jupyter-r.yazdi/.local/lib/python3.10/site-packages/albumentations/check_version.py:49: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


device: cuda:0


## Dataset & dataloaders\n(unchanged from `SegFormer_maxsig.ipynb`)

In [3]:
class mydataset(Dataset):
    def __init__(self, images_dir, masks_dir, transforms_img=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transforms_img = transforms_img
        self.mask_names = glob(masks_dir + "/*.tif") + glob(masks_dir + "/*.bmp") + glob(masks_dir + "/*.png")

    def __len__(self):
        return len(self.mask_names)

    def __getitem__(self, index):
        mask_name = self.mask_names[index]
        mask = cv2.imread(mask_name, cv2.IMREAD_ANYDEPTH)
        mask = (mask - mask.min()) / (mask.max() - mask.min())
        mask = (mask > 0).astype(np.float32)
        mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_AREA)

        imgn = mask_name.replace('/masks/', '/images/')
        if imgn.endswith('.tif'):
            if not os.path.exists(imgn):
                imgn = imgn.replace('.tif', '.bmp')

        image = cv2.imread(imgn, cv2.IMREAD_ANYDEPTH)
        image = (image - image.min()) / (image.max() - image.min()).astype(np.float32)
        image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)

        if self.transforms_img is not None:
            transformed = self.transforms_img(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        return image, mask


In [4]:
batch_size = config['batch_size']

train_transform = A.Compose([ToTensorV2()])
val_transform   = A.Compose([ToTensorV2()])
test_transform  = A.Compose([ToTensorV2()])

train_ds = mydataset(ROOT + 'dataset/train/images/', ROOT + 'dataset/train/masks/', train_transform)
val_ds   = mydataset(ROOT + 'dataset/val/images/',   ROOT + 'dataset/val/masks/',   val_transform)
test_ds  = mydataset(ROOT + 'dataset/test/images/',  ROOT + 'dataset/test/masks/',  test_transform)

train_dataset = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dataset   = DataLoader(val_ds,   batch_size=batch_size, shuffle=True)
test_dataset  = DataLoader(test_ds,  batch_size=batch_size, shuffle=True)

print("train batches:", len(train_dataset), "| val batches:", len(val_dataset), "| test batches:", len(test_dataset))


train batches: 37 | val batches: 8 | test batches: 8


## MaxSig components\n(unchanged from `SegFormer_maxsig.ipynb`)

In [8]:
class MaxSigLayerNormalized(nn.Module):
    def __init__(self, ch_in: int, ch_out: int, kernel_size: int, stride: int = 1, use_sigmoid_input: bool = True):
        super().__init__()
        assert kernel_size >= 1
        self.ch_in = ch_in
        self.ch_out = ch_out
        self.kernel_size = kernel_size
        self.stride = stride
        self.use_sigmoid_input = use_sigmoid_input

        self.padding = (kernel_size - 1) // 2
        self.kernel_area = kernel_size ** 2
        self.center_index = self.kernel_area // 2

        self.weight = nn.Parameter(torch.rand(1, 1, self.kernel_area, 1, 1))
        self.bias   = nn.Parameter(torch.zeros(1, 1, self.kernel_area, 1, 1))

        self.weight_center = nn.Parameter(torch.tensor(1.0))
        self.weight_median = nn.Parameter(torch.tensor(1.0))

        self.proj = nn.Conv2d(ch_in, ch_out, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C_in, H, W = x.shape
        assert C_in == self.ch_in, f"Expected {self.ch_in} channels, got {C_in}"

        x_in = torch.sigmoid(x) if self.use_sigmoid_input else x

        x_unfold = F.unfold(x_in, kernel_size=self.kernel_size, padding=self.padding, stride=self.stride)
        _, CK2, L = x_unfold.shape
        assert CK2 == C_in * self.kernel_area

        out_h = (H + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_w = (W + 2 * self.padding - self.kernel_size) // self.stride + 1

        x_unfold = x_unfold.view(B, C_in, self.kernel_area, out_h, out_w)

        w = torch.sigmoid(self.weight)
        b = torch.sigmoid(self.bias)

        D = torch.maximum(w, x_unfold) + b

        mean_stat = torch.mean(D, dim=2)
        median_stat, _ = torch.median(D, dim=2)

        center_values = x_unfold[:, :, self.center_index, :, :]

        out = (self.weight_center * center_values + self.weight_median * median_stat - mean_stat)
        out = self.proj(out)
        return out


class MaxSigGate(nn.Module):
    """Gated residual: x + gamma * post(maxsig(x)). gamma starts at 0 => baseline SegFormer
    at init, MaxSig mixed in as training proceeds. Same pattern as DownBlockWithMaxSig (U-Net)
    and GatedMaxSigWrapper (TransUNet)."""
    def __init__(self, channels: int, kernel_size: int = 3, gamma_init: float = 0.0,
                 use_sigmoid_input: bool = True):
        super().__init__()
        self.maxsig = MaxSigLayerNormalized(
            ch_in=channels, ch_out=channels, kernel_size=kernel_size,
            stride=1, use_sigmoid_input=use_sigmoid_input,
        )
        self.post = nn.Sequential(nn.BatchNorm2d(channels), nn.LeakyReLU(0.1, inplace=True))
        self.gamma = nn.Parameter(torch.tensor(gamma_init))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.maxsig(x)
        m = self.post(m)
        return x + self.gamma * m


## Mix Vision Transformer (MiT) encoder\n(unchanged from `SegFormer_maxsig.ipynb` — reimplemented from the SegFormer paper for version-stability; only `SegformerConfig`/`SegformerDecodeHead` are used from `transformers`)

In [10]:
class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size, stride):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size,
                               stride=stride, padding=patch_size // 2)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, H, W


class EfficientSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, sr_ratio, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.q = nn.Linear(dim, dim, bias=True)
        self.kv = nn.Linear(dim, dim * 2, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)
        self.sr_ratio = sr_ratio
        if sr_ratio > 1:
            self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
            self.norm = nn.LayerNorm(dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        q = self.q(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        if self.sr_ratio > 1:
            x_ = x.permute(0, 2, 1).reshape(B, C, H, W)
            x_ = self.sr(x_).reshape(B, C, -1).permute(0, 2, 1)
            x_ = self.norm(x_)
            kv = self.kv(x_)
        else:
            kv = self.kv(x)
        kv = kv.reshape(B, -1, 2, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = self.proj(out)
        out = self.proj_drop(out)
        return out


class DWConv(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, H, W)
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)
        return x


class MixFFN(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.dwconv = DWConv(hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x, H, W):
        x = self.fc1(x)
        x = self.dwconv(x, H, W)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class MixTransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio, sr_ratio, drop=0.0, attn_drop=0.0,
                 drop_path=0.0, layer_norm_eps=1e-6):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=layer_norm_eps)
        self.attn = EfficientSelfAttention(dim, num_heads, sr_ratio, attn_drop, drop)
        self.drop_path = DropPath(drop_path)
        self.norm2 = nn.LayerNorm(dim, eps=layer_norm_eps)
        self.mlp = MixFFN(dim, dim * mlp_ratio, drop)

    def forward(self, x, H, W):
        x = x + self.drop_path(self.attn(self.norm1(x), H, W))
        x = x + self.drop_path(self.mlp(self.norm2(x), H, W))
        return x


class MixVisionTransformerStage(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size, stride, depth, num_heads, mlp_ratio,
                 sr_ratio, drop=0.0, attn_drop=0.0, drop_path_rates=None, layer_norm_eps=1e-6):
        super().__init__()
        self.patch_embed = OverlapPatchEmbed(in_ch, embed_dim, patch_size, stride)
        drop_path_rates = drop_path_rates or [0.0] * depth
        self.blocks = nn.ModuleList([
            MixTransformerBlock(embed_dim, num_heads, mlp_ratio, sr_ratio, drop, attn_drop,
                                 drop_path_rates[j], layer_norm_eps)
            for j in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=layer_norm_eps)

    def forward(self, x):
        x, H, W = self.patch_embed(x)
        for blk in self.blocks:
            x = blk(x, H, W)
        x = self.norm(x)
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return x


class MixVisionTransformerEncoder(nn.Module):
    def __init__(self, config: SegformerConfig):
        super().__init__()
        depths = config.depths
        total_depth = sum(depths)
        dpr = [config.drop_path_rate * i / max(total_depth - 1, 1) for i in range(total_depth)]

        self.stages = nn.ModuleList()
        cur = 0
        in_ch = config.num_channels
        for i in range(config.num_encoder_blocks):
            stage_dpr = dpr[cur:cur + depths[i]]
            self.stages.append(MixVisionTransformerStage(
                in_ch=in_ch,
                embed_dim=config.hidden_sizes[i],
                patch_size=config.patch_sizes[i],
                stride=config.strides[i],
                depth=depths[i],
                num_heads=config.num_attention_heads[i],
                mlp_ratio=config.mlp_ratios[i],
                sr_ratio=config.sr_ratios[i],
                drop=config.hidden_dropout_prob,
                attn_drop=config.attention_probs_dropout_prob,
                drop_path_rates=stage_dpr,
                layer_norm_eps=config.layer_norm_eps,
            ))
            cur += depths[i]
            in_ch = config.hidden_sizes[i]

    def forward(self, x):
        hidden_states = []
        for stage in self.stages:
            x = stage(x)
            hidden_states.append(x)
        return hidden_states


## SegFormer + MaxSig model\n(unchanged from `SegFormer_maxsig.ipynb`)

In [11]:
class SegFormerMaxSig(nn.Module):
    def __init__(self, out_size=(256, 256), variant="b0", maxsig_stages=(1, 2),
                 maxsig_kernel=3, gamma_init=0.0, use_sigmoid_input=True):
        super().__init__()

        variants = {
            "b0": dict(depths=[2, 2, 2, 2],  hidden_sizes=[32, 64, 160, 256],  decoder_hidden_size=256),
            "b1": dict(depths=[2, 2, 2, 2],  hidden_sizes=[64, 128, 320, 512], decoder_hidden_size=256),
            "b2": dict(depths=[3, 4, 6, 3],  hidden_sizes=[64, 128, 320, 512], decoder_hidden_size=768),
            "b3": dict(depths=[3, 4, 18, 3], hidden_sizes=[64, 128, 320, 512], decoder_hidden_size=768),
            "b4": dict(depths=[3, 8, 27, 3], hidden_sizes=[64, 128, 320, 512], decoder_hidden_size=768),
            "b5": dict(depths=[3, 6, 40, 3], hidden_sizes=[64, 128, 320, 512], decoder_hidden_size=768),
        }
        preset = variants[variant]

        seg_config = SegformerConfig(
            num_channels=1,
            num_labels=1,
            depths=preset["depths"],
            hidden_sizes=preset["hidden_sizes"],
            decoder_hidden_size=preset["decoder_hidden_size"],
        )

        self.encoder = MixVisionTransformerEncoder(seg_config)
        self.decode_head = SegformerDecodeHead(seg_config)
        self.out_size = out_size

        maxsig_stages = set(maxsig_stages)
        self.maxsig_gates = nn.ModuleDict()
        for i in range(seg_config.num_encoder_blocks):
            if i in maxsig_stages:
                self.maxsig_gates[str(i)] = MaxSigGate(
                    channels=seg_config.hidden_sizes[i],
                    kernel_size=maxsig_kernel,
                    gamma_init=gamma_init,
                    use_sigmoid_input=use_sigmoid_input,
                )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden_states = x
        encoder_hidden_states = []
        for i, stage in enumerate(self.encoder.stages):
            hidden_states = stage(hidden_states)
            if str(i) in self.maxsig_gates:
                hidden_states = self.maxsig_gates[str(i)](hidden_states)
            encoder_hidden_states.append(hidden_states)

        logits = self.decode_head(tuple(encoder_hidden_states))
        logits = F.interpolate(logits, size=self.out_size, mode="bilinear", align_corners=False)
        return logits


## Ablation helper

`run_scenario(run_name, maxsig_stages, ...)` builds a fresh `SegFormerMaxSig` with MaxSig gates at
the given encoder stages (`()` = no MaxSig anywhere), trains it, evaluates test Dice with the best
checkpoint, and reports only `best_val_dice`, `best_epoch`, `test_dice`.


In [12]:
def dice_from_logits(preds_logits, labels):
    pred_sig = torch.sigmoid(preds_logits)
    preds = (pred_sig > 0.5).float()
    intersection = (preds * labels).sum()
    union = (preds + labels).sum()
    return (2 * intersection / union)


def run_scenario(run_name, maxsig_stages, epochs=config['epochs'], verbose_every=10):
    torch.manual_seed(42)

    model = SegFormerMaxSig(
        variant=config['segformer_variant'],
        maxsig_stages=maxsig_stages,
        maxsig_kernel=config['maxsig_kernel'],
        gamma_init=config['gamma_init'],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    scheduler = StepLR(optimizer, step_size=100, gamma=0.83)
    scaler = torch.amp.GradScaler('cuda')

    ckpt_path = os.path.join(RESULTS_DIR, f"best-{run_name}.pt")
    best_valid_dice = 0.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        for inputs, mask in train_dataset:
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda"):
                outputs = model(inputs.to(device))
                labels = mask.to(device).unsqueeze(1)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        dice_score = 0.0
        with torch.no_grad():
            model.eval()
            for inputs, mask in val_dataset:
                inputs = inputs.to(device)
                preds = model(inputs)
                labels = mask.to(device).unsqueeze(1)
                dice_score += dice_from_logits(preds, labels)

        epoch_dc = float(dice_score / len(val_dataset))
        if best_valid_dice < epoch_dc:
            best_valid_dice = epoch_dc
            best_epoch = epoch
            torch.save(model.state_dict(), ckpt_path)

        if verbose_every and (epoch + 1) % verbose_every == 0:
            print(f"[{run_name}] epoch {epoch+1}/{epochs} | val_dice={epoch_dc:.4f} "
                  f"| best={best_valid_dice:.4f}@{best_epoch+1}")

    # ---- test evaluation with the best checkpoint ----
    model.load_state_dict(torch.load(ckpt_path))
    dice_score = 0.0
    with torch.no_grad():
        model.eval()
        for inputs, mask in test_dataset:
            inputs = inputs.to(device)
            preds = model(inputs)
            labels = mask.to(device).unsqueeze(1)
            dice_score += dice_from_logits(preds, labels)
    test_dice = float(dice_score / len(test_dataset))

    result = {
        "run_name": run_name,
        "maxsig_stages": tuple(sorted(maxsig_stages)),
        "best_val_dice": best_valid_dice,
        "best_epoch": best_epoch + 1,
        "test_dice": test_dice,
    }
    print(f"[{run_name}] DONE | best_val_dice={best_valid_dice:.4f} (epoch {best_epoch+1}) "
          f"| test_dice={test_dice:.4f}")
    return result


ablation_results = []


### Scenario 1 — Default: MaxSig at stages `(1, 2)` — 64ch + 160ch\n(the configuration already used in `SegFormer_maxsig.ipynb`)

In [9]:
res = run_scenario(run_name="default_stages_1_2", maxsig_stages=(1, 2))
ablation_results.append(res)


/tmp/ipykernel_4182275/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[default_stages_1_2] epoch 10/50 | val_dice=0.6759 | best=0.6759@10
[default_stages_1_2] epoch 20/50 | val_dice=0.7239 | best=0.7239@20
[default_stages_1_2] epoch 30/50 | val_dice=0.6862 | best=0.7282@26
[default_stages_1_2] epoch 40/50 | val_dice=0.6937 | best=0.7282@26
[default_stages_1_2] epoch 50/50 | val_dice=0.6901 | best=0.7341@41
[default_stages_1_2] DONE | best_val_dice=0.7341 (epoch 41) | test_dice=0.6623


### Scenario 2 — MaxSig at **Stage 0 only (32ch, highest resolution)**
> Reviewer framing: *"Did you test applying MaxSig at the earliest, highest-resolution encoder
> stage and find it hurt performance?"* — the SegFormer analogue of the Stage-0 (64ch) question
> raised for the plain U-Net.


In [10]:
res = run_scenario(run_name="stage0_only_32ch", maxsig_stages=(0,))
ablation_results.append(res)


/tmp/ipykernel_4182275/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[stage0_only_32ch] epoch 10/50 | val_dice=0.6545 | best=0.6811@9
[stage0_only_32ch] epoch 20/50 | val_dice=0.6710 | best=0.7151@16
[stage0_only_32ch] epoch 30/50 | val_dice=0.7029 | best=0.7184@26
[stage0_only_32ch] epoch 40/50 | val_dice=0.7013 | best=0.7319@32
[stage0_only_32ch] epoch 50/50 | val_dice=0.7144 | best=0.7319@32
[stage0_only_32ch] DONE | best_val_dice=0.7319 (epoch 32) | test_dice=0.6185


### Scenario 3 — MaxSig at all four encoder stages `(0, 1, 2, 3)`
> Reviewer framing: *"Did you test adding MaxSig to every encoder stage, including the deepest,
> attention-heavy one, to see whether the benefit compounds or performance degrades?"*


In [11]:
res = run_scenario(run_name="all_stages_0_1_2_3", maxsig_stages=(0, 1, 2, 3))
ablation_results.append(res)


/tmp/ipykernel_4182275/1748191668.py:14: RuntimeWarning: invalid value encountered in divide
  mask = (mask - mask.min()) / (mask.max() - mask.min())


[all_stages_0_1_2_3] epoch 10/50 | val_dice=0.6027 | best=0.6424@9
[all_stages_0_1_2_3] epoch 20/50 | val_dice=0.6619 | best=0.7199@19
[all_stages_0_1_2_3] epoch 30/50 | val_dice=0.7275 | best=0.7275@30
[all_stages_0_1_2_3] epoch 40/50 | val_dice=0.7093 | best=0.7275@30
[all_stages_0_1_2_3] epoch 50/50 | val_dice=0.7125 | best=0.7275@30
[all_stages_0_1_2_3] DONE | best_val_dice=0.7275 (epoch 30) | test_dice=0.6441


## Summary table

In [12]:
df = pd.DataFrame(ablation_results)
df = df[["run_name", "maxsig_stages", "best_val_dice", "best_epoch", "test_dice"]]
df


,run_name,maxsig_stages,best_val_dice,best_epoch,test_dice
0,default_stages_1_2,"(1, 2)",0.734057,41,0.662256
1,stage0_only_32ch,"(0,)",0.731896,32,0.618466
2,all_stages_0_1_2_3,"(0, 1, 2, 3)",0.727511,30,0.644115


In [13]:

def compute_flops_fps(model, input_size=(1, 1, 512, 512), n_warmup=10, n_iters=50):
    """Reports GFLOPs (thop) and FPS (measured wall-clock throughput) for a model."""
    model.eval()
    dummy = torch.randn(*input_size).to(device)

    try:
        from thop import profile
        macs, params = profile(model, inputs=(dummy,), verbose=False)
        gflops = macs * 2 / 1e9
    except ImportError:
        print("`thop` not installed. Run `pip install thop` to get GFLOPs. Skipping GFLOPs.")
        gflops = None

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()
        for _ in range(n_iters):
            _ = model(dummy)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = perf_counter()

    fps = n_iters / (t1 - t0)
    return gflops, fps




In [15]:
default_model = SegFormerMaxSig(
        variant=config['segformer_variant'],
        maxsig_stages=(1,2),
        maxsig_kernel=config['maxsig_kernel'],
        gamma_init=config['gamma_init'],
    ).to(device)
gflops, fps = compute_flops_fps(default_model, input_size=(1, 1, 256, 256))
print(f"Default config (stage=(2,)) | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


Default config (stage=(2,)) | GFLOPs: 3.379 | FPS: 388.01


In [17]:
baseline_model = SegFormerMaxSig(
        variant=config['segformer_variant'],
        maxsig_stages=(),
        maxsig_kernel=config['maxsig_kernel'],
        gamma_init=config['gamma_init'],
    ).to(device)
gflops, fps = compute_flops_fps(baseline_model, input_size=(1, 1, 256, 256))
print(f"baseline | GFLOPs: {gflops if gflops is None else round(gflops, 3)} | FPS: {fps:.2f}")


baseline | GFLOPs: 3.356 | FPS: 446.05
